In [14]:
healthy = ["/dsi/efroni-lab/sbm/chen2/raw_data/healthy/TRB_h_v1", "/dsi/efroni-lab/sbm/chen2/raw_data/healthy/TRB_h_v2"]
sick = "/dsi/efroni-lab/sbm/chen2/raw_data/sick/mixcr_4.7_TRB"
downsample_path = "/dsi/efroni-lab/sbm/chen2/downsampled_data"

In [15]:
def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered

In [16]:
import pandas as pd
from glob import glob
from collections import Counter
import chephy_model as cpm
from tqdm import tqdm

# ---------- CONFIG ----------
cdr3_header = 'aaSeqCDR3'  # Adjust if your column is different


# ---------- 1. Load patient files ----------
def load_patient_files(paths, cdr3_header='CDR3'):
    patient_dfs = []
    for path in paths if isinstance(paths, list) else [paths]:
        files = glob(f"{path}/*.tsv")
        for f in tqdm(files):
            df = pd.read_csv(f, sep='\t', low_memory=False)
            df['patient_id'] = f  # tag patient by file path
            patient_dfs.append(df)
    return patient_dfs

# Replace with your folder paths
healthy_patients = load_patient_files(healthy, cdr3_header)
sick_patients = load_patient_files(sick, cdr3_header)

# ---------- 2. Prepare + Truncate BEFORE public sequence detection ----------
def prepare_and_truncate(df, cdr3_header):
    df = prepare_data(df, cdr3_header)
    df = cpm.truncate_sequences(df, cdr3_header, 4, 4)
    return df

healthy_prepared = [prepare_and_truncate(df, cdr3_header) for df in healthy_patients]
sick_prepared = [prepare_and_truncate(df, cdr3_header) for df in sick_patients]
all_prepared = healthy_prepared + sick_prepared

cdr3_header_truncated = "cdr3_truncated"  # Update header after truncation
# ---------- 3. Identify public sequences (after truncation) ----------
seq_patient_map = {}
for df in all_prepared:
    pid = df['patient_id'].iloc[0]
    for seq in df[cdr3_header_truncated].dropna().unique():
        seq_patient_map.setdefault(seq, set()).add(pid)

public_seqs = {seq for seq, pids in seq_patient_map.items() if len(pids) > 1}

100%|██████████| 47/47 [00:06<00:00,  7.47it/s]


In [17]:
downsample_size = 15000

# ---------- 4. Final patient processing ----------
def process_patient(df, cdr3_header, public_seqs, sample=True):
    df = prepare_data(df, cdr3_header)
    df = cpm.truncate_sequences(df, cdr3_header, 4, 4)
    df = df[~df['cdr3_truncated'].isin(public_seqs)]
    df = df.drop_duplicates(subset='cdr3_truncated')
    
    if not sample:
        return df if len(df) > 0 else None
    
    if len(df) >= downsample_size:
        return df.sample(downsample_size, random_state=42, weights='readFraction')
    else:
        return None

full = True

# ---------- 5. Process healthy/sick datasets ----------
healthy_processed = [process_patient(df, cdr3_header, public_seqs, sample=full) for df in healthy_patients]
sick_processed = [process_patient(df, cdr3_header, public_seqs, sample=full) for df in sick_patients]


# ---------- 6. Filter out patients with <8000 usable sequences ----------
healthy_processed = [df for df in healthy_processed if df is not None]
sick_processed = [df for df in sick_processed if df is not None]


In [18]:
len(healthy_processed), len(sick_processed)

(32, 21)

In [19]:
import os

output_dir = f"{downsample_path}/{downsample_size}"
os.makedirs(output_dir, exist_ok=True)

def save_processed_patients(processed_list, label):
    for i, df in enumerate(processed_list):

        pid = os.path.basename(str(df['patient_id'].iloc[0]))
        out_path = os.path.join(output_dir, f"{label}_{pid}")
        df.to_csv(out_path, sep='\t', index=False)
        print(f"Saved {label} patient {i} to {out_path}")

save_processed_patients(healthy_processed, "healthy")
save_processed_patients(sick_processed, "sick")

Saved healthy patient 0 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_27.clones_TRB.tsv
Saved healthy patient 1 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_39.clones_TRB.tsv
Saved healthy patient 2 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_30.clones_TRB.tsv
Saved healthy patient 3 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_50.clones_TRB.tsv
Saved healthy patient 4 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_33.clones_TRB.tsv
Saved healthy patient 5 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_46.clones_TRB.tsv
Saved healthy patient 6 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_31.clones_TRB.tsv
Saved healthy patient 7 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_42.clones_TRB.tsv
Saved healthy patient 8 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/healthy_19.clones_TRB.tsv
Saved healthy patient 9 to /dsi/efroni-lab/sbm/chen2/downsampled_data/15000/health

In [20]:
import pandas as pd

df = pd.read_feather("../data/big_data/serumdetect_tcr_data.2024-09-20.ftr")

FileNotFoundError: [Errno 2] No such file or directory: '../data/big_data/serumdetect_tcr_data.2024-09-20.ftr'